# Phase 2 — Word Representations
Uses the Phase-1 output `data/processed/cleaned_reviews.parquet` **unchanged** (393,891 cleaned Amazon Fine Food reviews).
Embedding models are trained ONCE by `python -m src.build_representations`; this notebook only **loads** them.

In [1]:
import sys, json, math, time
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))
import numpy as np, pandas as pd, scipy.sparse as sp
import matplotlib; matplotlib.use('Agg'); import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
FIG = ROOT/'figures'
pd.set_option('display.width', 160); pd.set_option('display.max_colwidth', 90)
df = pd.read_parquet(ROOT/'data'/'processed'/'cleaned_reviews.parquet')
texts = df['clean_text']
print(f"documents: {len(df):,}   tokens: {texts.str.split().str.len().sum():,}")
def sparsity(X): return 100 * (1 - X.nnz / (X.shape[0] * X.shape[1]))
SPLIT = dict(tokenizer=str.split, token_pattern=None, lowercase=False)   # text is already cleaned

documents: 393,891   tokens: 15,192,247


## Part A — Count-based representations
### A1. Bag of Words

In [2]:
bow_full = CountVectorizer(**SPLIT)                 # every type, no frequency cut-off
X_bow = bow_full.fit_transform(texts)
print(f"BoW (all types)  shape = {X_bow.shape}   non-zeros = {X_bow.nnz:,}   sparsity = {sparsity(X_bow):.4f}%")
print(f"memory as sparse CSR  = {(X_bow.data.nbytes + X_bow.indices.nbytes + X_bow.indptr.nbytes)/1e6:.0f} MB "
      f"(dense int64 would be {X_bow.shape[0]*X_bow.shape[1]*8/1e9:,.0f} GB)")
print(f"avg distinct words per review = {X_bow.nnz / X_bow.shape[0]:.1f} of {X_bow.shape[1]:,}")
bow5 = CountVectorizer(min_df=5, **SPLIT); X_bow5 = bow5.fit_transform(texts)
print(f"BoW (min_df=5, saved for Phase 3) shape = {X_bow5.shape}   sparsity = {sparsity(X_bow5):.4f}%")
counts = np.asarray(X_bow.sum(axis=0)).ravel(); vocab = bow_full.get_feature_names_out()
print('most frequent columns:', [(vocab[i], int(counts[i])) for i in counts.argsort()[::-1][:12]])

BoW (all types)  shape = (393891, 90596)   non-zeros = 12,125,925   sparsity = 99.9660%
memory as sparse CSR  = 147 MB (dense int64 would be 285 GB)
avg distinct words per review = 30.8 of 90,596


BoW (min_df=5, saved for Phase 3) shape = (393891, 24963)   sparsity = 99.8777%
most frequent columns: [('not', 438120), ('but', 257322), ('like', 192278), ('taste', 186313), ('flavor', 141003), ('good', 140366), ('product', 132285), ('love', 119877), ('just', 119515), ('very', 117459), ('great', 116002), ('try', 114345)]


### A2. TF-IDF — high vs low weights on a real review

In [3]:
tfidf_uni = TfidfVectorizer(**SPLIT)                # sklearn defaults: idf = ln((1+n)/(1+df)) + 1, L2-normalised rows
X_tfidf = tfidf_uni.fit_transform(texts)
print(f"TF-IDF shape = {X_tfidf.shape}   sparsity = {sparsity(X_tfidf):.4f}%")
idf = pd.Series(tfidf_uni.idf_, index=tfidf_uni.get_feature_names_out())
dfreq = pd.Series(np.asarray((X_bow > 0).sum(axis=0)).ravel(), index=vocab)
print('\nLowest idf (in the most reviews):'); print(pd.DataFrame({'df': dfreq[idf.nsmallest(8).index], 'idf': idf.nsmallest(8).round(3)}).T.to_string())

TF-IDF shape = (393891, 90596)   sparsity = 99.9660%



Lowest idf (in the most reviews):
            not         but        like       taste        good       love      great       just
df   221641.000  169188.000  127902.000  126469.000  107721.000  97677.000  94563.000  92264.000
idf       1.575       1.845       2.125       2.136       2.297      2.394      2.427      2.451


In [4]:
# One concrete review (Id 8)
i = df.index[df['Id'] == 8][0]
print('RAW  :', df.loc[i, 'Text']); print('CLEAN:', df.loc[i, 'clean_text']); print()
row = X_tfidf[i]; cnt = X_bow[i]
terms = tfidf_uni.get_feature_names_out()[row.indices]
tab = pd.DataFrame({'term': terms,
                    'tf (count)': [cnt[0, bow_full.vocabulary_[t]] for t in terms],
                    'df (reviews)': [int(dfreq[t]) for t in terms],
                    'idf': [round(idf[t], 3) for t in terms],
                    'tf-idf (L2-normed)': row.data.round(4)}).sort_values('tf-idf (L2-normed)', ascending=False)
print(tab.to_string(index=False))

RAW  : This taffy is so good.  It is very soft and chewy.  The flavors are amazing.  I would definitely recommend you buying it.  Very satisfying!!
CLEAN: taffy good very soft chewy flavor amaze definitely recommend buy very satisfy

      term  tf (count)  df (reviews)   idf  tf-idf (L2-normed)
     taffy           1           397 7.897              0.5244
     chewy           1          5407 5.288              0.3512
      very           2         90210 2.474              0.3286
   satisfy           1          9000 4.779              0.3173
     amaze           1          9850 4.689              0.3113
      soft           1          9948 4.679              0.3107
definitely           1         18671 4.049              0.2689
 recommend           1         37118 3.362              0.2232
       buy           1         87389 2.506              0.1664
    flavor           1         92195 2.452              0.1628
      good           1        107721 2.297              0.1525


### A3. N-grams — vocabulary growth and sparsity

In [5]:
sample = texts.sample(100_000, random_state=42)
rows = []
for rng in [(1,1), (2,2), (3,3), (1,2), (1,3)]:
    for mindf in [1, 5]:
        t = time.time(); v = CountVectorizer(ngram_range=rng, min_df=mindf, **SPLIT); X = v.fit_transform(sample)
        rows.append({'ngram_range': str(rng), 'min_df': mindf, 'features': X.shape[1], 'non-zeros': X.nnz,
                     'nnz / doc': round(X.nnz/X.shape[0], 1), 'sparsity %': round(sparsity(X), 4),
                     'singleton features %': round(100*np.mean(np.asarray((X>0).sum(0)).ravel()==1), 1) if mindf==1 else None})
        del X, v
ng = pd.DataFrame(rows); print(ng.to_string(index=False))

ngram_range  min_df  features  non-zeros  nnz / doc  sparsity %  singleton features %
     (1, 1)       1     45744    3081195       30.8     99.9326                  49.0
     (1, 1)       5     13435    3032788       30.3     99.7743                   NaN
     (2, 2)       1   1181202    3664747       36.6     99.9969                  70.1
     (2, 2)       5    110903    2223487       22.2     99.9800                   NaN
     (3, 3)       1   2981409    3644514       36.4     99.9988                  91.0
     (3, 3)       5     36509     383608        3.8     99.9895                   NaN
     (1, 2)       1   1226946    6745942       67.5     99.9945                  69.3
     (1, 2)       5    124338    5256275       52.6     99.9577                   NaN
     (1, 3)       1   4208355   10390456      103.9     99.9975                  84.7
     (1, 3)       5    160847    5639883       56.4     99.9649                   NaN


In [6]:
sub = ng[ng['ngram_range'].isin(['(1, 1)', '(1, 2)', '(1, 3)'])]
fig, ax = plt.subplots(figsize=(5.6, 3.2))
for mindf, mk in [(1, 'o-'), (5, 's--')]:
    d = sub[sub.min_df == mindf]; ax.plot(['1', '1–2', '1–3'], d['features'], mk, label=f'min_df={mindf}')
    for x, y in zip(['1', '1–2', '1–3'], d['features']): ax.annotate(f'{y:,}', (x, y), textcoords='offset points', xytext=(4, 4), fontsize=7)
ax.set_yscale('log'); ax.set_xlabel('n-gram range'); ax.set_ylabel('features (log)'); ax.set_title('Feature count vs n (100k-review sample)'); ax.legend()
plt.tight_layout(); plt.savefig(FIG/'p2_fig1_ngram_growth.png', dpi=150); print('saved')

saved


### A4. Hand-worked numerical on 4 real reviews

In [7]:
toy_ids = [169230, 270144, 199681, 243558]
toy = df.set_index('Id').loc[toy_ids, ['Text', 'clean_text']]
docs = toy['clean_text'].tolist()
for k, (rid, r) in enumerate(toy.iterrows(), 1): print(f"D{k} (Id {rid}): {r['clean_text']!r}   <- {r['Text']!r}")

D1 (Id 169230): 'great coffee hard find local store'   <- 'Great Coffee  which is hard to find in local stores'
D2 (Id 270144): 'great coffee like ice coffee love brand'   <- 'This is a great coffee.  If you like iced coffee then to will love this brand'
D3 (Id 199681): 'okay not good pumpkin'   <- 'These were Ok not as good as some of the other pumpkins I have had.'
D4 (Id 243558): 'very good coffee roast bite light taste'   <- 'Very good coffee -- perhaps roasted a bit on the light side for my taste.'


In [8]:
# ---- TF-IDF of 'coffee' in D2, by hand (sklearn default: smooth idf, raw tf, L2 norm) ----
n = len(docs); tok = [d.split() for d in docs]
def df_(w): return sum(w in d for d in tok)
def idf_(w): return math.log((1 + n) / (1 + df_(w))) + 1
d2 = tok[1]
print(f"n = {n};  D2 = {d2}")
print(f"tf(coffee, D2) = {d2.count('coffee')}   df(coffee) = {df_('coffee')}  (D1, D2, D4)")
print(f"idf(coffee) = ln((1+{n})/(1+{df_('coffee')})) + 1 = ln({(1+n)/(1+df_('coffee')):.4f}) + 1 = {idf_('coffee'):.4f}")
raw = {w: d2.count(w) * idf_(w) for w in dict.fromkeys(d2)}
for w, v in raw.items(): print(f"   raw  {w:7}: tf={d2.count(w)} x idf={idf_(w):.4f} = {v:.4f}")
norm = math.sqrt(sum(v*v for v in raw.values()))
print(f"||D2|| = sqrt({' + '.join(f'{v:.4f}^2' for v in raw.values())}) = {norm:.4f}")
hand = raw['coffee'] / norm
print(f"tfidf(coffee, D2) = {raw['coffee']:.4f} / {norm:.4f} = {hand:.4f}")
tv = TfidfVectorizer(**SPLIT); Xt = tv.fit_transform(docs)
lib = Xt[1, tv.vocabulary_['coffee']]
print(f"scikit-learn       = {lib:.4f}   match: {math.isclose(hand, lib, rel_tol=1e-9)}")
print(f"(compare: 'great' in D2 = {raw['great']/norm:.4f}, 'brand' in D2 = {raw['brand']/norm:.4f})")

n = 4;  D2 = ['great', 'coffee', 'like', 'ice', 'coffee', 'love', 'brand']
tf(coffee, D2) = 2   df(coffee) = 3  (D1, D2, D4)
idf(coffee) = ln((1+4)/(1+3)) + 1 = ln(1.2500) + 1 = 1.2231
   raw  great  : tf=1 x idf=1.5108 = 1.5108
   raw  coffee : tf=2 x idf=1.2231 = 2.4463
   raw  like   : tf=1 x idf=1.9163 = 1.9163
   raw  ice    : tf=1 x idf=1.9163 = 1.9163
   raw  love   : tf=1 x idf=1.9163 = 1.9163
   raw  brand  : tf=1 x idf=1.9163 = 1.9163
||D2|| = sqrt(1.5108^2 + 2.4463^2 + 1.9163^2 + 1.9163^2 + 1.9163^2 + 1.9163^2) = 4.7912
tfidf(coffee, D2) = 2.4463 / 4.7912 = 0.5106
scikit-learn       = 0.5106   match: True
(compare: 'great' in D2 = 0.3153, 'brand' in D2 = 0.4000)


In [9]:
# ---- Bigram by hand: count of 'great coffee' and P(coffee | good) ----
bigrams = [list(zip(d, d[1:])) for d in tok]
for k, b in enumerate(bigrams, 1): print(f"D{k} bigrams: {[' '.join(x) for x in b]}")
c_gc = sum(b.count(('great', 'coffee')) for b in bigrams)
print(f"\ncount('great coffee') over the 4 docs = {c_gc}  (D1 + D2)")
bv = CountVectorizer(ngram_range=(2, 2), **SPLIT); Xb = bv.fit_transform(docs)
lib_c = int(Xb[:, bv.vocabulary_['great coffee']].sum())
print(f"CountVectorizer(ngram_range=(2,2)) column 'great coffee' = {Xb[:, bv.vocabulary_['great coffee']].toarray().ravel()} -> total {lib_c}   match: {lib_c == c_gc}")
c_good = sum(d.count('good') for d in tok); c_goodcoffee = sum(b.count(('good', 'coffee')) for b in bigrams)
print(f"P(coffee | good) = count(good coffee) / count(good) = {c_goodcoffee} / {c_good} = {c_goodcoffee/c_good:.2f}")

D1 bigrams: ['great coffee', 'coffee hard', 'hard find', 'find local', 'local store']
D2 bigrams: ['great coffee', 'coffee like', 'like ice', 'ice coffee', 'coffee love', 'love brand']
D3 bigrams: ['okay not', 'not good', 'good pumpkin']
D4 bigrams: ['very good', 'good coffee', 'coffee roast', 'roast bite', 'bite light', 'light taste']

count('great coffee') over the 4 docs = 2  (D1 + D2)
CountVectorizer(ngram_range=(2,2)) column 'great coffee' = [1 1 0 0] -> total 2   match: True
P(coffee | good) = count(good coffee) / count(good) = 1 / 2 = 0.50


## Part B — Learned embeddings (loaded from `models/`, trained by `src/build_representations.py`)

In [10]:
from gensim.models import Word2Vec, FastText
log = json.loads((ROOT/'models'/'training_log.json').read_text())['embeddings']
print('hyper-parameters (identical for all three):', json.dumps(log['params']))
print('fastText extra:', log['fasttext_extra'])
print(f"training corpus: {log['documents']:,} reviews, {log['tokens']:,} tokens")
cbow = Word2Vec.load(str(ROOT/'models'/'w2v_cbow.model')).wv
skip = Word2Vec.load(str(ROOT/'models'/'w2v_skipgram.model')).wv
ft   = FastText.load(str(ROOT/'models'/'fasttext.model')).wv
M = {'CBOW': cbow, 'Skip-gram': skip, 'fastText': ft}
print(pd.DataFrame.from_dict({k: {'sg': (0 if k == 'CBOW' else 1), 'vocab': len(v), 'dim': v.vector_size, 'train seconds': log[f]['seconds']}
                    for (k, v), f in zip(M.items(), ['w2v_cbow', 'w2v_skipgram', 'fasttext'])}, orient='index').to_string())

hyper-parameters (identical for all three): {"vector_size": 100, "window": 5, "min_count": 5, "epochs": 5, "negative": 5, "sample": 0.001, "workers": 2, "seed": 42}
fastText extra: {'min_n': 3, 'max_n': 6, 'bucket': 500000}
training corpus: 393,891 reviews, 15,192,247 tokens


           sg  vocab  dim  train seconds
CBOW        0  26442  100           56.3
Skip-gram   1  26442  100          208.1
fastText    1  26442  100          310.7


### B4. Rare / out-of-vocabulary / misspelled words: Word2Vec vs fastText

In [11]:
from collections import Counter
freq = Counter(t for s in texts for t in s.split())
queries = ['cofee', 'expresso', 'deliciouss', 'cofeeee', 'chocolatte', 'flavorfull']   # frequent typos, rare words, unseen words
for q in queries:
    print(f"\n'{q}'  corpus count = {freq.get(q, 0)}   in Word2Vec vocab: {q in skip.key_to_index}")
    if q in skip.key_to_index: print('   Skip-gram:', [(w, round(s, 2)) for w, s in skip.most_similar(q, topn=5)])
    else: print('   Skip-gram: KeyError -> no vector at all')
    print('   fastText :', [(w, round(s, 2)) for w, s in ft.most_similar(q, topn=5)])


'cofee'  corpus count = 100   in Word2Vec vocab: True
   Skip-gram: [('coffee', 0.81), ('esspresso', 0.8), ('coffer', 0.79), ('caffee', 0.78), ('davidoff', 0.78)]
   fastText : [('cofffee', 0.89), ('coffeee', 0.88), ('coffee', 0.87), ('coffe', 0.87), ('coffes', 0.85)]

'expresso'  corpus count = 525   in Word2Vec vocab: True
   Skip-gram: [('espresso', 0.92), ('coffee', 0.81), ('americanos', 0.79), ('lungo', 0.79), ('intenso', 0.78)]
   fastText : [('espresso', 0.89), ('esspresso', 0.87), ('nesspresso', 0.87), ('redespresso', 0.85), ('iperespresso', 0.84)]

'deliciouss'  corpus count = 2   in Word2Vec vocab: False
   Skip-gram: KeyError -> no vector at all
   fastText : [('deliciouse', 0.96), ('delicious', 0.95), ('deliciously', 0.91), ('deeeelicious', 0.9), ('deliciuos', 0.89)]

'cofeeee'  corpus count = 0   in Word2Vec vocab: False
   Skip-gram: KeyError -> no vector at all
   fastText : [('cofee', 0.94), ('coffeee', 0.88), ('cofffee', 0.87), ('coffe', 0.83), ('coffeepot', 0.83)]

'

In [12]:
# how fastText builds an OOV vector: sum of its character n-gram vectors (3..6, with < > boundary marks)
from gensim.models.fasttext import ft_ngram_hashes
w = '<cofee>'
grams = [w[i:i+n] for n in range(3, 7) for i in range(len(w)-n+1)]
print(len(grams), 'n-grams of', w, ':', grams)
print('cosine(fastText["cofee"], fastText["coffee"]) =', round(float(ft.similarity('cofee', 'coffee')), 3))

14 n-grams of <cofee> : ['<co', 'cof', 'ofe', 'fee', 'ee>', '<cof', 'cofe', 'ofee', 'fee>', '<cofe', 'cofee', 'ofee>', '<cofee', 'cofee>']
cosine(fastText["cofee"], fastText["coffee"]) = 0.871


## Part C — Probing & comparison
### C1. Nearest neighbours

In [13]:
probe = ['coffee', 'delicious', 'stale', 'dog', 'chocolate']
for q in probe:
    print(f"\n=== {q}")
    tab = pd.DataFrame({name: [f"{w} ({s:.2f})" for w, s in kv.most_similar(q, topn=8)] for name, kv in M.items()})
    print(tab.to_string(index=False))


=== coffee
             CBOW         Skip-gram           fastText
     coffe (0.89)      coffe (0.90)     coffeee (0.95)
  expresso (0.83)  columbian (0.83)       coffe (0.94)
     cofee (0.79)     coffer (0.82)     cofffee (0.93)
  espresso (0.79)   expresso (0.81)      coffes (0.92)
 starbucks (0.78)      cofee (0.81)     coffess (0.91)
cappuccino (0.76)   espresso (0.80)   coffeepot (0.90)
 columbian (0.73)       java (0.78)  coffeeshop (0.89)
 cappucino (0.72) cappaccino (0.78) coffeeshops (0.88)

=== delicious
              CBOW          Skip-gram            fastText
      yummy (0.82)       yummy (0.85)   deliciouse (0.94)
      tasty (0.76)       tasty (0.84)  deeelicious (0.91)
  wonderful (0.76)    fabulous (0.82) deeeelicious (0.91)
   fabulous (0.75) scrumptious (0.82)     delicous (0.88)
scrumptious (0.74)    delicous (0.81)  deliciously (0.87)
  fantastic (0.72)   wonderful (0.80)      licious (0.86)
     delish (0.72)   fantastic (0.79)   dellicious (0.85)
   delicous (0

### C2. Cosine similarities

In [14]:
pairs = [('coffee', 'espresso'), ('delicious', 'tasty'), ('coffee', 'dog'), ('delicious', 'ship'), ('good', 'bad')]
sim = pd.DataFrame({name: [round(float(kv.similarity(a, b)), 3) for a, b in pairs] for name, kv in M.items()},
                   index=[f'{a} – {b}' for a, b in pairs])
sim['expected'] = ['similar', 'similar', 'unrelated', 'unrelated', 'antonyms']
print(sim.to_string())

                    CBOW  Skip-gram  fastText   expected
coffee – espresso  0.791      0.805     0.813    similar
delicious – tasty  0.759      0.836     0.851    similar
coffee – dog      -0.160      0.161     0.167  unrelated
delicious – ship  -0.037      0.258     0.306  unrelated
good – bad         0.637      0.693     0.689   antonyms


### C3. Analogies (a − b + c ≈ ?)

In [15]:
analogies = [('puppy', 'dog', 'cat'),        # dog : puppy :: cat : ? (kitten)
             ('espresso', 'coffee', 'tea'),  # coffee : espresso :: tea : ? (matcha?)
             ('bark', 'dog', 'cat')]         # dog : bark :: cat : ? (meow/purr)
for a, b, c in analogies:
    print(f"\n{a} - {b} + {c} = ?")
    for name, kv in M.items():
        print(f"   {name:9}", [(w, round(s, 2)) for w, s in kv.most_similar(positive=[a, c], negative=[b], topn=5)])


puppy - dog + cat = ?
   CBOW      [('kitten', 0.93), ('kitty', 0.84), ('siamese', 0.78), ('felidae', 0.74), ('female', 0.72)]
   Skip-gram [('kitten', 0.9), ('kitty', 0.82), ('coon', 0.78), ('ragdoll', 0.77), ('siamese', 0.77)]
   fastText  [('kitten', 0.92), ('kitteh', 0.9), ('kittenhood', 0.86), ('kitty', 0.84), ('kittys', 0.8)]

espresso - coffee + tea = ?
   CBOW      [('sencha', 0.66), ('rooibos', 0.65), ('infusion', 0.59), ('matcha', 0.58), ('matchas', 0.57)]
   Skip-gram [('rooibos', 0.72), ('sencha', 0.7), ('twinnings', 0.7), ('rishi', 0.68), ('tazo', 0.68)]
   fastText  [('senchas', 0.76), ('rooibos', 0.75), ('sencha', 0.73), ('rishi', 0.71), ('twinnings', 0.7)]

bark - dog + cat = ?
   CBOW      [('mulch', 0.61), ('shrub', 0.56), ('scamper', 0.53), ('clipping', 0.53), ('bury', 0.53)]
   Skip-gram [('catnip', 0.54), ('eucalyptus', 0.51), ('elm', 0.5), ('mound', 0.5), ('valarian', 0.5)]
   fastText  [('tilia', 0.56), ('catnips', 0.55), ('balck', 0.54), ('catnip', 0.54), ('rab

In [16]:
# 2-D PCA of a few word groups (Skip-gram)
from sklearn.decomposition import PCA
groups = {'drinks': ['coffee', 'tea', 'espresso', 'latte', 'cocoa', 'decaf'],
          'pets': ['dog', 'cat', 'puppy', 'kitten', 'treat', 'kibble'],
          'taste +': ['delicious', 'tasty', 'yummy', 'excellent', 'great'],
          'taste −': ['stale', 'bland', 'awful', 'gross', 'rancid']}
words = [w for g in groups.values() for w in g if w in skip.key_to_index]
P = PCA(2, random_state=0).fit_transform(np.array([skip[w] for w in words]))
fig, ax = plt.subplots(figsize=(5.6, 4))
cols = dict(zip(groups, ['#4C78A8', '#F58518', '#54A24B', '#E45756']))
for (x, y), w in zip(P, words):
    g = next(k for k, v in groups.items() if w in v); ax.scatter(x, y, color=cols[g], s=18)
    ax.annotate(w, (x, y), fontsize=7.5, xytext=(3, 2), textcoords='offset points')
for g, c in cols.items(): ax.scatter([], [], color=c, label=g)
ax.legend(fontsize=7, loc='best'); ax.set_title('Skip-gram vectors, PCA to 2-D'); ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout(); plt.savefig(FIG/'p2_fig2_pca.png', dpi=150); print('saved')

saved


### C4. Count-based vs learned — measured on this corpus

In [17]:
tf_saved = __import__('joblib').load(ROOT/'models'/'tfidf_vectorizer.joblib')
Xs = tf_saved.transform(texts.iloc[:20000])
toks20 = [d.split() for d in texts.iloc[:20000]]
oov_tok = np.mean([t not in skip.key_to_index for d in toks20 for t in d])
oov_doc = np.mean([any(t not in skip.key_to_index for t in d) for d in toks20])
comp = pd.DataFrame({
  'BoW (min_df=5)':        [X_bow5.shape[1], f"{sparsity(X_bow5):.3f}%", 'dropped', 'which words occur (no order)'],
  'TF-IDF 1–2 gram (saved)': [len(tf_saved.vocabulary_), f"{sparsity(Xs):.3f}%", 'dropped', 'distinctive words + short phrases ("not good")'],
  'Word2Vec CBOW/SG':      [cbow.vector_size, '0% (dense)', 'no vector (skipped)', 'distributional meaning; similar words close'],
  'fastText':              [ft.vector_size, '0% (dense)', 'built from char n-grams', 'meaning + spelling / morphology'],
}, index=['dimensions', 'sparsity', 'unseen word', 'captures']).T
print(comp.to_string())
print(f"\nfirst 20k reviews: {100*oov_tok:.2f}% of tokens are OOV for Word2Vec; {100*oov_doc:.1f}% of reviews contain at least one OOV token")

                        dimensions    sparsity              unseen word                                        captures
BoW (min_df=5)               24963     99.878%                  dropped                    which words occur (no order)
TF-IDF 1–2 gram (saved)     405293     99.986%                  dropped  distinctive words + short phrases ("not good")
Word2Vec CBOW/SG               100  0% (dense)      no vector (skipped)     distributional meaning; similar words close
fastText                       100  0% (dense)  built from char n-grams                 meaning + spelling / morphology

first 20k reviews: 0.60% of tokens are OOV for Word2Vec; 15.5% of reviews contain at least one OOV token


## Part D — Saved models and the `vectorize()` interface

In [18]:
import os
for f in sorted((ROOT/'models').glob('*')):
    if f.name != '.gitkeep': print(f"{f.name:32} {os.path.getsize(f)/1e6:8.1f} MB")
from src.vectorize import vectorize, METHODS
doc = "I didn't like this coffee at all!! Way too bitter <br />and overpriced."
print('\nraw doc:', doc)
for m in METHODS:
    t = time.time(); v = vectorize(doc, m); dt = time.time() - t
    desc = f"sparse {v.shape}, nnz={v.nnz}" if sp.issparse(v) else f"dense {v.shape}, first 4 = {np.round(v[:4], 3)}"
    print(f"vectorize(doc, '{m}'): {desc}   ({dt:.2f}s incl. first load)")
print('\nnon-zero TF-IDF features:', sorted(zip(np.round(vectorize(doc, 'tfidf').data, 3),
      tf_saved.get_feature_names_out()[vectorize(doc, 'tfidf').indices]), reverse=True))

bow_vectorizer.joblib                 0.3 MB
fasttext.model                       22.0 MB
fasttext.model.wv.vectors_ngrams.npy    200.0 MB
tfidf_vectorizer.joblib              11.0 MB
training_log.json                     0.0 MB
w2v_cbow.model                       22.0 MB
w2v_skipgram.model                   22.0 MB



raw doc: I didn't like this coffee at all!! Way too bitter <br />and overpriced.
vectorize(doc, 'bow'): sparse (1, 24963), nnz=7   (0.18s incl. first load)


vectorize(doc, 'tfidf'): sparse (1, 405293), nnz=12   (0.52s incl. first load)
vectorize(doc, 'cbow'): dense (100,), first 4 = [-1.709  0.29   0.286 -0.089]   (0.10s incl. first load)


vectorize(doc, 'skipgram'): dense (100,), first 4 = [-0.167 -0.035 -0.196 -0.011]   (0.10s incl. first load)


vectorize(doc, 'fasttext'): dense (100,), first 4 = [-0.163  0.095 -0.066 -0.051]   (0.74s incl. first load)

non-zero TF-IDF features: [(np.float64(0.473), 'coffee way'), (np.float64(0.418), 'too bitter'), (np.float64(0.378), 'overpriced'), (np.float64(0.332), 'like coffee'), (np.float64(0.33), 'way too'), (np.float64(0.259), 'bitter'), (np.float64(0.238), 'not like'), (np.float64(0.192), 'way'), (np.float64(0.174), 'coffee'), (np.float64(0.164), 'too'), (np.float64(0.117), 'like'), (np.float64(0.086), 'not')]
